# Check Skill — Notebook Control

두 가지 자동 검증:
1. `install()` — ManiSkill 환경 헬스체크 (imports / sim / 렌더)
2. `dataset(path)` — 생성된 HDF5 구조/통계 검증

두 호출 패턴 모두 검증.


## Setup — import path

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)

project root: C:\Users\hun41\PycharmProjects\maniskill


## ① install() — Function mode

In [2]:
from scripts.check import install

result = install()
print('overall passed:', result['passed'])
for c in result['checks']:
    mark = 'PASS' if c['passed'] else 'FAIL'
    print(f"  [{mark}] {c['name']}: {c['detail']}")

C:\Users\hun41\miniconda3\envs\maniskill\Lib\site-packages\sapien\wrapper\pinocchio_model.py:299: UserWarning: pinnochio package is not installed, robotics functionalities will not be available
  warnings.warn(


overall passed: True
  [PASS] imports: mani_skill=3.0.1, sapien=3.0.3, torch=2.12.0+cpu
  [PASS] cuda_available_info: torch.cuda.is_available()=False
  [PASS] env_construct_reset: PickCube-v1 cpu/cpu reset OK
  [PASS] cpu_render: frame shape=(512, 512, 3), mean px=94.6
  [PASS] demos_present: tasks downloaded: ['PickCube-v1']


## ② dataset(path) — Function mode

In [3]:
from scripts.check import dataset

h5_path = (PROJECT_ROOT / 'data' / 'datasets' / 'PickCube-v1'
           / 'trajectory.rgb.pd_joint_pos.physx_cpu.h5')
result = dataset(h5_path)
print('overall passed:', result['passed'])
for c in result['checks']:
    mark = 'PASS' if c['passed'] else 'FAIL'
    print(f"  [{mark}] {c['name']}: {c['detail']}")

overall passed: True
  [PASS] file_exists: C:\Users\hun41\PycharmProjects\maniskill\data\datasets\PickCube-v1\trajectory.rgb.pd_joint_pos.physx_cpu.h5 (1.12 MB)
  [PASS] has_episodes: 3 episodes
  [PASS] required_fields_present: actions/obs/success all present
  [PASS] episode_lengths: min/mean/max steps = 50/66.0/74
  [PASS] success_rate: 3/3 = 100.0%
  [PASS] rgb_shape_consistent: camera 'base_camera' frames (128, 128, 3) dtype uint8


## ③ Subprocess mode — CLI 호출 동등성 확인

In [4]:
import subprocess, sys as _sys

result = subprocess.run(
    [_sys.executable, str(PROJECT_ROOT / 'scripts' / 'check.py'),
     '--dataset', str(h5_path)],
    capture_output=True, text=True,
)
print('returncode:', result.returncode, '(0=pass, 1=fail)')
print(result.stdout.strip())

returncode: 0 (0=pass, 1=fail)
=== Dataset Check: C:\Users\hun41\PycharmProjects\maniskill\data\datasets\PickCube-v1\trajectory.rgb.pd_joint_pos.physx_cpu.h5 ===
  [PASS] file_exists  -- C:\Users\hun41\PycharmProjects\maniskill\data\datasets\PickCube-v1\trajectory.rgb.pd_joint_pos.physx_cpu.h5 (1.12 MB)
  [PASS] has_episodes  -- 3 episodes
  [PASS] required_fields_present  -- actions/obs/success all present
  [PASS] episode_lengths  -- min/mean/max steps = 50/66.0/74
  [PASS] success_rate  -- 3/3 = 100.0%
  [PASS] rgb_shape_consistent  -- camera 'base_camera' frames (128, 128, 3) dtype uint8

Overall: PASS
